# EDA — Mất cân bằng nhãn BI-RADS gốc & gộp 3 nhóm hành động

Thống kê số lượng mẫu theo **nhãn gốc** (`breast_birads`) của dữ liệu VinDr-Mammo — 
**không** phải nhãn dự đoán từ mô hình:
- **5 lớp BI-RADS gốc:** BI-RADS 1, 2, 3, 4, 5
- **3 nhóm hành động (sau gộp):** Normal (BI-RADS 1), Benign-leaning (BI-RADS 2–3), 
  Suspicious (BI-RADS 4–5) — đúng mapping `_BIRADS_TO_LABEL` của notebook huấn luyện
  `bestone_v3_cv_oof.ipynb`

Biểu đồ cột ghép ở cuối notebook cho thấy rõ sự **mất cân bằng mẫu giữa 5 lớp gốc** 
và sự **phân bố lại khi gộp thành 3 nhóm hành động**.


In [ ]:
# =============================================================================
# CELL 1 — IMPORTS & REPRODUCIBILITY
# =============================================================================
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

_GLOBAL_SEED = 42
np.random.seed(_GLOBAL_SEED)


In [ ]:
# =============================================================================
# CELL 2 — NẠP DỮ LIỆU (NHÃN GỐC TỪ VINDR-MAMMO)
# =============================================================================
# Dùng đúng 2 CSV như notebook huấn luyện (bestone_v3_cv_oof.ipynb). Nếu chạy
# ngoài Kaggle (file không tồn tại) → fallback sang phân bố tài liệu hóa để
# notebook vẫn vẽ được biểu đồ minh họa.

_DETECTION_CSV = "/kaggle/input/datasets/phanxunkhoa/mammovin/vindr_detection_v1_folds.csv"
_METADATA_CSV  = "/kaggle/input/datasets/phanxunkhoa/mammovin/metadata.csv"

# Phân bố tài liệu hóa (fallback): 5 lớp BI-RADS gốc, tổng = 4098 mẫu
_FALLBACK_5CLASS_COUNTS = {
    "BI-RADS 1": 2778,
    "BI-RADS 2": 848,
    "BI-RADS 3": 196,
    "BI-RADS 4": 167,
    "BI-RADS 5": 109,
}

def load_vindr_counts():
    """Trả về {nhãn BI-RADS gốc -> số mẫu}, tính từ metadata thật hoặc fallback."""
    if os.path.exists(_DETECTION_CSV) and os.path.exists(_METADATA_CSV):
        df_det = pd.read_csv(_DETECTION_CSV, low_memory=False)
        df_meta = pd.read_csv(_METADATA_CSV, low_memory=False)
        df_det["image_id_clean"] = df_det["image_id"].str.replace(".png", "", regex=False)
        df = pd.merge(df_det, df_meta, left_on="image_id_clean",
                      right_on="SOP Instance UID", how="inner")
        df = df.drop_duplicates(subset=["image_id"]).reset_index(drop=True)
        counts = df["breast_birads"].value_counts().to_dict()
        counts = {k: int(v) for k, v in counts.items() if k in _FALLBACK_5CLASS_COUNTS}
        print(f"✅ Dữ liệu thật: {len(df)} mẫu (dedup theo image_id, nhãn gốc breast_birads)")
        return counts
    print("⚠️ Không tìm thấy CSV trên Kaggle — dùng phân bố tài liệu hóa để minh họa.")
    return dict(_FALLBACK_5CLASS_COUNTS)


In [ ]:
# =============================================================================
# CELL 3 — BẢNG SỐ LIỆU: 5 LỚP GỐC & 3 NHÓM HÀNH ĐỘNG
# =============================================================================
order5  = ["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5"]
counts5 = load_vindr_counts()
counts5 = {k: counts5.get(k, 0) for k in order5}

# Bản đồ gộp 5 lớp → 3 nhóm hành động (map dùng khi huấn luyện mô hình)
group_of = {
    "BI-RADS 1": "Normal (BI-RADS 1)",
    "BI-RADS 2": "Benign-leaning (BI-RADS 2-3)",
    "BI-RADS 3": "Benign-leaning (BI-RADS 2-3)",
    "BI-RADS 4": "Suspicious (BI-RADS 4-5)",
    "BI-RADS 5": "Suspicious (BI-RADS 4-5)",
}
order3 = ["Normal (BI-RADS 1)", "Benign-leaning (BI-RADS 2-3)", "Suspicious (BI-RADS 4-5)"]

df5 = pd.DataFrame({"label": order5, "count": [counts5[k] for k in order5]})
df5["pct"] = df5["count"] / df5["count"].sum() * 100

group_counts = {}
for k, v in counts5.items():
    group_counts[group_of[k]] = group_counts.get(group_of[k], 0) + v
df3 = pd.DataFrame({"group": order3, "count": [group_counts[g] for g in order3]})
df3["pct"] = df3["count"] / df3["count"].sum() * 100

print("───── 5 lớp BI-RADS gốc (nhãn gốc) ─────")
print(df5.to_string(index=False))
print()
print("───── 3 nhóm hành động (sau gộp) ─────")
print(df3.to_string(index=False))


## Biểu đồ cột ghép (Grouped Bar Chart)

Cụm trái = **5 lớp BI-RADS gốc**; cụm phải = **3 nhóm hành động** (tổng của các thành 
phần cùng hành động). Màu cột được giữ nhất quán theo nhóm sau gộp để thấy rõ cách gộp: 
BI-RADS 1 → Normal, BI-RADS 2–3 → Benign-leaning, BI-RADS 4–5 → Suspicious.


In [ ]:
# =============================================================================
# CELL 4 — GROUPED BAR CHART: 5 LỚP GỐC vs 3 NHÓM HÀNH ĐỘNG
# =============================================================================
_COLOR_GROUP = {
    "Normal (BI-RADS 1)":           "#2E86AB",
    "Benign-leaning (BI-RADS 2-3)": "#F5B041",
    "Suspicious (BI-RADS 4-5)":     "#D7263D",
}

fig, ax = plt.subplots(figsize=(16, 6.2))
fig.subplots_adjust(right=0.82, top=0.86, bottom=0.14)

# --- Cụm A: 5 lớp gốc ở vị trí 0..4 ---
x5 = np.arange(5)
colors5 = [_COLOR_GROUP[group_of[k]] for k in order5]
ax.bar(x5, df5["count"], width=0.62, color=colors5,
       edgecolor="black", linewidth=0.6)
for x_, c, p in zip(x5, df5["count"], df5["pct"]):
    ax.text(x_, c, f"{int(c):,}\n({p:.1f}%)", ha="center", va="bottom", fontsize=10)

# --- Cụm B: 3 nhóm hành động ở vị trí 6..8 (ngăn cách bằng khoảng trống) ---
x3 = np.arange(6, 9)
colors3 = [_COLOR_GROUP[g] for g in order3]
ax.bar(x3, df3["count"], width=0.62, color=colors3,
       edgecolor="black", linewidth=0.6)
for x_, c, p in zip(x3, df3["count"], df3["pct"]):
    ax.text(x_, c, f"{int(c):,}\n({p:.1f}%)", ha="center", va="bottom", fontsize=10)

# --- Trục + nhãn nhóm ---
ax.set_xticks(list(range(9)))
ax.set_xticklabels(["BI-RADS 1", "BI-RADS 2", "BI-RADS 3", "BI-RADS 4", "BI-RADS 5",
                    "", "Normal\n(BI-RADS 1)", "Benign-leaning\n(BI-RADS 2-3)", "Suspicious\n(BI-RADS 4-5)"])
ax.axvline(5.5, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ymax = max(df5["count"].max(), df3["count"].max())
ax.text(2.0, ymax * 1.10, "5 lớp BI-RADS gốc", ha="center", fontsize=13, fontweight="bold")
ax.text(7.0, ymax * 1.10, "3 nhóm hành động\n(sau gộp)", ha="center", fontsize=13, fontweight="bold")

ax.set_ylabel("Số lượng mẫu (nhãn gốc)", fontsize=11)
ax.set_ylim(0, ymax * 1.30)
ax.grid(axis="y", alpha=0.3)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

# --- Legend đặt NGOÀI mép phải (không đè lên tiêu đề) ---
legend_handles = [
    plt.Rectangle((0, 0), 1, 1, color=_COLOR_GROUP[g], label=g) for g in order3
]
ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1.0),
          frameon=True, title="Nhóm hành động", title_fontsize=10, fontsize=10)

fig.suptitle("Grouped Bar Chart — Mất cân bằng 5 lớp BI-RADS gốc và phân bố lại 3 nhóm hành động",
             fontsize=14, fontweight="bold", y=0.99)
plt.show()
